# Dataset 3 Setup, Loading, and Cleaning

#### Setup and imports
Python Virtual Environment Commands

In [73]:
# using a virtual environment ensures that all the packages you install (pandas, numpy, etc.)
# are isolated from your system Python. this helps your project work consistently whether
# you run it on your own laptop or in another environment. it's completely fine if you don't want
# to set this up, you can still run the notebook without it.

# create a virtual environment named 'venv':
# python3 -m venv venv

# activate the virtual environment (Windows or macOS/Linux):
# venv\Scripts\activate
# source venv/bin/activate

# once in the virtual environment, install the right packages:
# pip install pandas numpy matplotlib seaborn

# run the notebook as usual

# deactivate the virtual environment:
# deactivate

In [74]:
# data handling
import pandas as pd   # primary library for loading, inspecting, cleaning, and manipulating tabular data
import numpy as np    # numerical computing library, useful for calculations and handling missing values

# visualization (optional but helpful during cleaning)
import matplotlib.pyplot as plt  # basic plotting library for creating line charts, histograms, and scatterplots
import seaborn as sns            # higher-level visualization library for statistical plots and easier styling

# file handling / os operations
import os  # provides utilities to navigate directories, check file paths, and manage file locations

# optional: warnings suppression for a cleaner notebook
import warnings
warnings.filterwarnings('ignore')  # suppresses unnecessary warnings to make notebook outputs cleaner

# starter settings
pd.set_option('display.max_columns', None)  # show all columns when printing a dataframe
pd.set_option('display.max_rows', 20)      # limit number of rows displayed to avoid clutter
pd.set_option('display.float_format', '{:.2f}'.format)  # format floating numbers to 2 decimals

sns.set_style('whitegrid')  # makes plots cleaner and more readable

#### Loading Raw Dataset - [NYC Permitted Event Information](https://data.cityofnewyork.us/City-Government/NYC-Permitted-Event-Information/tvpp-9vvx/about_data)



In [75]:
# Loads the dataset into a pandas DataFrame
dataset3raw = pd.read_csv('../../data/raw/NYC_Permitted_Event_Information_Current.csv')  
# Displays the first few rows to verify import happened
dataset3raw.head()  

,Event ID,Event Name,Start Date/Time,End Date/Time,Event Agency,Event Type,Event Borough,Event Location,Event Street Side,Street Closure Type,Community Board,Police Precinct,CEMSID
0,907992,Soccer - Non Regulation,03/23/2026 04:00:00 PM,03/23/2026 07:00:00 PM,Parks Department,Sport - Youth,Queens,Hinton Park: Baseball-01,NaN,NaN,"3,","115,","6779,"
1,905553,Softball - Adults,04/01/2026 06:00:00 PM,04/01/2026 08:00:00 PM,Parks Department,Sport - Adult,Brooklyn,McCarren Park: Softball-01,NaN,NaN,"01,","94,","5106,"
2,897098,Miscellaneous,06/01/2026 09:00:00 AM,06/01/2026 01:00:00 PM,Parks Department,Special Event,Queens,Alley Pond Park: Alley Springfield BBQ Area Pink,NaN,NaN,"11,","111,","7575,"
3,913893,Softball - Adults,04/01/2026 04:00:00 PM,04/01/2026 06:00:00 PM,Parks Department,Sport - Youth,Brooklyn,Remsen Playground (PS 114): Softball-01,NaN,NaN,"18,","69,","8257,"
4,889684,Lawn Closure - East 106th St South Mount Lawn,04/02/2026 12:00:00 AM,04/02/2026 11:59:00 PM,Parks Department,Special Event,Manhattan,Central Park: East 106th St Lawn,NaN,NaN,"64,","22,","3697,"


#### Inspect Dataset for if it needs to be cleaned or not

In [76]:
# Look at the general information of the datset to understand its setup and structure
dataset3raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 26472 entries, 0 to 26471
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Event ID             26472 non-null  int64
 1   Event Name           26450 non-null  str  
 2   Start Date/Time      26472 non-null  str  
 3   End Date/Time        26472 non-null  str  
 4   Event Agency         26472 non-null  str  
 5   Event Type           26472 non-null  str  
 6   Event Borough        26472 non-null  str  
 7   Event Location       26472 non-null  str  
 8   Event Street Side    386 non-null    str  
 9   Street Closure Type  444 non-null    str  
 10  Community Board      26469 non-null  str  
 11  Police Precinct      26469 non-null  str  
 12  CEMSID               26079 non-null  str  
dtypes: int64(1), str(12)
memory usage: 2.6 MB


In [77]:
# Total datapoints in the dataset
total_datapoints = len(dataset3raw)

# Checks for the amount of missing values in each column
missingValsCount = dataset3raw.isnull().sum()
print("Missing values data percentage:\n" + str(missingValsCount / total_datapoints * 100), end="")

# Checks for the amount of duplicated rows in the dataset
duplicatedValsCount = (dataset3raw.duplicated().sum())
print("Duplicated rows data percentage:", duplicatedValsCount / total_datapoints * 100)

Missing values data percentage:
Event ID               0.00
Event Name             0.08
Start Date/Time        0.00
End Date/Time          0.00
Event Agency           0.00
Event Type             0.00
Event Borough          0.00
Event Location         0.00
Event Street Side     98.54
Street Closure Type   98.32
Community Board        0.01
Police Precinct        0.01
CEMSID                 1.48
dtype: float64Duplicated rows data percentage: 0.0


Conclusion Reached:

Any column with "Date/Time" should be converted from string to datetime to allow for easier handling later on. 

There are no duplicate rows but there are null values primarily in "Event Street Side" and "Street Closure Type." An unexpected discovery was that many of these data entries do not indicate a street closure, which means this data will have to eventually be mapped with some API to determine the surronding streets of the event venue so we can determine the impact of said events on traffic. Ultimately, I believe these columns should be eliminated from the dataset since they will only create more trouble and don't contribute much. Instead, I will convert null values to be an empty string for the time being in case we do need them later on.

#### Dataset Cleaning

In [78]:
# Adjust the columsn involving dates and times to be datetime objects 
dataset3raw['Start Date/Time'] = pd.to_datetime(dataset3raw['Start Date/Time'])
dataset3raw['End Date/Time'] = pd.to_datetime(dataset3raw['End Date/Time'])

# Converts the CEMSID column to numeric, coercing errors to NaN, then fills 
# those NaN values with 0 and converts the column to integers
# Note: This column may not ultimately be needed since this seems to be an internal ID for the event
dataset3raw['CEMSID'] = pd.to_numeric(dataset3raw['CEMSID'], errors='coerce')
dataset3raw['CEMSID'] = dataset3raw['CEMSID'].fillna(0).astype(int)

# Converts the Police Precinct column to numeric, coercing errors to NaN, then fills
# those NaN values with 0 and converts the column to integers
dataset3raw['Police Precinct'] = pd.to_numeric(dataset3raw['Police Precinct'], errors='coerce')
dataset3raw['Police Precinct'] = dataset3raw['Police Precinct'].fillna(0).astype(int)

# Cleans Event Street Side and Street Closure Type to have empty strings instead of null values and converts them to string type
dataset3raw['Event Street Side'] = dataset3raw['Event Street Side'].fillna('').astype(str)
dataset3raw['Street Closure Type'] = dataset3raw['Street Closure Type'].fillna('').astype(str)

# Cleans community board and event name to have empty strings instead of null values and converts them to string type
dataset3raw['Community Board'] = dataset3raw['Community Board'].fillna('').astype(str)
dataset3raw['Event Name'] = dataset3raw['Event Name'].fillna('').astype(str)

dataset3raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 26472 entries, 0 to 26471
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Event ID             26472 non-null  int64         
 1   Event Name           26472 non-null  str           
 2   Start Date/Time      26472 non-null  datetime64[us]
 3   End Date/Time        26472 non-null  datetime64[us]
 4   Event Agency         26472 non-null  str           
 5   Event Type           26472 non-null  str           
 6   Event Borough        26472 non-null  str           
 7   Event Location       26472 non-null  str           
 8   Event Street Side    26472 non-null  str           
 9   Street Closure Type  26472 non-null  str           
 10  Community Board      26472 non-null  str           
 11  Police Precinct      26472 non-null  int64         
 12  CEMSID               26472 non-null  int64         
dtypes: datetime64[us](2), int64(3), str(8)
mem

Exports cleaned data to data/processed 

In [79]:
dataset3raw.to_csv("../../data/processed/dataset3_cleaned.csv", index=False)

In [80]:
print(min(dataset3raw['Start Date/Time']), max(dataset3raw['Start Date/Time']))

2024-06-30 08:00:00 2026-12-27 08:00:00
